# Source Git Code

https://github.com/frankpd/nyc_geodatabase/blob/master/census_zbp/zbp_to_zcta.ipynb

Request an API key here: https://api.census.gov/data/key_signup.html

US Census Slack channel: https://join.slack.com/t/uscensusbureau/shared_invite/zt-250allqpg-0NJKOyv8A4AMMLo_JC7W5w 
(Link expires 30 days from October 12, 2023)

In [ ]:
import pandas as pd, requests, sqlite3, os, json
from IPython.display import clear_output
import io
import csv

## The following code makes an API request to retrieve ZBP data for a designated NAICS code. 
## Note: The user running the code needs to update file paths. The data that is retrieved comes in the form of a text file and is saved into the "inputs" folder. Subsequent code writes the data to csv files into the "outputs2" folder.
### https://api.census.gov/data/2018/zbp/variables.html
### https://api.census.gov/data/2017/zbp/variables.html
### https://api.census.gov/data/2016/zbp/variables.html

In [ ]:
# Initialize variables for API request
dsource = 'zbp'
ecols = 'ESTAB,EMP,EMPSZES,GEO_ID,ZIPCODE,PAYQTR1,PAYANN'
api_key = 'da5ca696289b328bc3575a3405c6196767cb452c'
naics = '445310' # Note: You can update the NAICS code to import other industry data

In [ ]:
# Updated to change census year depending on calendar year of data

# Define the path to the "input" folder relative to the user's home directory. 
# This is where the data retrieved from the API request will be stored.
user_home = os.path.expanduser("~")
input_folder = os.path.join(user_home, 'OneDrive - Colostate', 'Teaching', 'Courses', 'CSU', 'AREC570', 'api_docs', 'zbp_data', 'input')
print(input_folder)

# Define the mapping of years to nyear
year_to_nyear = {
    2000: 1997,
    2001: 1997,
    2002: 2002,
    2003: 2002,
    2004: 2002,
    2005: 2002,
    2006: 2002,
    2007: 2007,
    2008: 2007,
    2009: 2007,
    2010: 2007,
    2011: 2007,
    2012: 2012,
    2013: 2012,
    2014: 2012,
    2015: 2012,
    2016: 2012,
    2017: 2017,
    2018: 2017
}

# Loop through years and set nyear accordingly
for year in range(2000, 2019):
    nyear = year_to_nyear[year]
    base_url = f'https://api.census.gov/data/{year}/{dsource}'
    edata_url = f'{base_url}?get={ecols}&NAICS{nyear}={naics}&for=zipcode:*&key={api_key}'
    
    # Use edata_url for the desired API request or further processing
    print(f'For year {year}, nyear is {nyear}, and edata_url is: {edata_url}')
    
    # Make an API request, retrieve data, and save to a file
    a = requests.get(edata_url)
    quote = a.text
    outpath = os.path.join(input_folder, f'Out{year}_NAICS{naics}.txt')
    with open(outpath, 'w', encoding='utf-8') as f:
        f.write(quote)
        
    print(outpath)

In [ ]:
# Specify the input folder and output folder
user_home = os.path.expanduser("~")
output_folder = os.path.join(user_home, 'OneDrive - Colostate', 'Teaching', 'Courses', 'CSU', 'AREC570', 'api_docs', 'zbp_data', 'output')
print(output_folder)

# Function to remove brackets from a list
def remove_brackets(lst):
    return [item.strip('[]"') for item in lst]

# Loop through the text files in the input folder
for filename in os.listdir(input_folder):
    if filename.endswith(".txt"):
        # Construct the full file paths
        input_file = os.path.join(input_folder, filename)
        output_file = os.path.join(output_folder, os.path.splitext(filename)[0] + ".csv")

        # Open the input text file for reading
        with open(input_file, 'r', newline='', encoding='utf-8') as txt_file:
            # Read the data from the text file and remove brackets
            data = [remove_brackets(row) for row in csv.reader(txt_file)]
            
            # Print the top 5 rows
            print(f'Top 5 rows of {filename}:')
            for row in data[:5]:
                print(row)

            # Open the corresponding output CSV file for writing
            with open(output_file, 'w', newline='', encoding='utf-8') as csv_file:
                # Write the data to the CSV file
                csv.writer(csv_file).writerows(data)

        print(f'Converted {filename} to {os.path.basename(output_file)}')

print("Conversion complete.")
